In [ ]:
import Pkg; Pkg.add(["SpecialFunctions"])


In [ ]:
using Random, Distributions, SpecialFunctions


# Parameters

This section sets the model's fundamentals and structural parameters, and fixes the indexing convention used throughout the rest of the notebook.

**Regions and markets.** There are $N$ regions and $J$ real sectors, plus a non-employment "sector 0" in every region, giving $M = J+1$ markets per region. Any array indexed over every possible market a household could occupy (labor $L$, value function $V$, migration shares $\mu$, mobility costs $\tau$) is `N × M`, with **column 1 = non-employment** and **columns 2:M = real sectors 1..J**. Arrays that only pertain to production ($A$, $w$, $\kappa$, $\theta$, $\eta$, $\gamma$, $\alpha$) are `N × J`, since sector 0 has no production, wage, or price — to compare a market-indexed array against a sector-indexed one, offset the sector index by $+1$.

**Time-varying fundamentals**, $\Theta_t = (A_t, \kappa_t)$:
- $A_t^{nj}$ — productivity in region $n$, sector $j$
- $\kappa_t^{nj,ij}$ — iceberg trade cost shipping sector-$j$ goods from region $i$ to region $n$

**Constant fundamentals**, $\bar\Theta = (\Upsilon, b)$:
- $\Upsilon = \{\tau^{nj,ik}\}$ — labor relocation (mobility) costs, in utility terms, from market $(n,j)$ to market $(i,k)$
- $b^n$ — value of home production in region $n$ (a non-employed household's consumption)

**Structural parameters:**
- $\beta \in [0,1)$ — discount factor
- $\theta^j$ — Fréchet trade elasticity in sector $j$
- $\nu$ — dispersion of the idiosyncratic migration taste shock ($1/\nu$ is the migration elasticity)
- $\alpha^j$ — Cobb-Douglas consumption share on sector $j$, with $\sum_j \alpha^j = 1$

**State variable:** $L_t = \{L_t^{nj}\}$, the mass of households in each market at time $t$ — the only object carrying information from one period to the next.

In [ ]:
#try indexing format of J[region]_[sector]_[time], x if not indexed by that component

N = 2 # Number of regions
J = 3 # Number of real sectors (sector 0 / non-employment is handled separately -- see M below)
M = J + 1 # Number of markets per region: non-employment (market column 1, i.e. sector 0) plus
          # the J real sectors (market columns 2:M, i.e. sectors 1:J). Per the model, a "market"
          # is a region-sector pair (n,j) with j = 0,...,J, so any array indexed over "every
          # market a household could be in" (L, V, mu, tau_mig) has rows = regions (N), columns
          # = markets (M), with column 1 = non-employment. Arrays that only pertain to real
          # production (A, w, kappa, theta, eta, gamma, alpha) keep rows = regions, columns =
          # real sectors (J) -- to look one of these up against a market-indexed array, offset
          # the sector index by +1 (real sector j lives in market column j+1).
n_omega = 10000 # number of varieties used to discretize the omega in [0,1] continuum per region-sector


L_0  = ones(N,M) #labor force in economy at time 0, over all N regions x M markets (incl. non-employment)

Random.seed!(1) # Seed for the random number generator (to guarantee reproducibility; this is standard in research these days)

A_0 = rand(N,J) #region-sector productivity (real sectors only; undefined for non-employment)

B = ones(N,J) #arbitrary coefficient

w_0 = ones(N,J) # wage, real sectors only -- non-employment pays no wage; households there consume b_n instead

b = ones(N) # value of home production: consumption of a non-employed household in region n

kappa_0 = [ones(N,N) for _ in 1:J] # iceberg trade costs, per sector (kappa: trade cost)

# sigma = 2 # Substitution elasticity between goods
# L = ones(N, 1) # Size of labor force in each country

theta = fill(4.0, J) # Frechet shape parameter per sector (governs dispersion of productivity draws
                      # across the continuum of varieties/regions, and the trade elasticity);
                      # scale is absorbed into A_0, so no separate T parameter is needed

eta = fill(2.0, N, J) # elasticity of substitution across varieties within sector j (CES aggregator)

gamma = ones(N, J) # labor/value-added share of production; = 1 since this simplified model has no materials

beta = 0.95 # household discount factor
# utility cost of moving from market (n,j) to market (i,k), j,k = 0,...,J (market columns 1:M,
# where column 1 is non-employment); 0 to stay in the same market, 1 otherwise (tau: migration cost)
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J) # Cobb-Douglas consumption shares: alpha[g] is the share of household income
alpha = alpha ./ sum(alpha) # spent on good g, common across all regions/sectors; normalized to sum to 1

nu = 1.0 # dispersion of the idiosyncratic (Frechet/type-I EV) taste shock over migration destinations
         # (larger nu = migration is less sensitive to utility differences, i.e. more friction)

T = 10 # number of periods to simulate the labor dynamics forward


# Production - Intermediates

This section implements the static production block, for real sectors $j=1,\dots,J$ only (sector 0 has no production side).

**Production.** A firm producing a variety with efficiency draw $z$ in market $(n,j)$ has constant returns in labor alone:
$$q_t^{nj}(z) = z \cdot A_t^{nj} \cdot l_t^{nj}$$
Productivity draws are discretized over $n_\omega$ varieties, drawn i.i.d. from a standard Fréchet with shape $\theta^j$; labor is split evenly across varieties within a region-sector (this doesn't affect aggregates, since unit cost is identical across all varieties in the same $(n,j)$).

**Unit cost.** With no materials or structures, the unit cost of producing one unit of the variety collapses to just the wage (scaled by the placeholder coefficient $B$, currently $\equiv 1$):
$$x_t^{nj} = B^{nj} \cdot w_t^{nj}$$

**Trade shares (gravity equation).** Each variety is sourced from whichever region is cheapest after trade costs. The resulting probability that region $n$ sources sector-$j$ goods from region $i$:
$$\pi_t^{nj,ij} = \frac{\left(x_t^{ij}\kappa_t^{nj,ij}\right)^{-\theta^j}\left(A_t^{ij}\right)^{\theta^j}}{\sum_{m=1}^N \left(x_t^{mj}\kappa_t^{nj,mj}\right)^{-\theta^j}\left(A_t^{mj}\right)^{\theta^j}}$$
`trade_cost_term[n,j,i]` is exactly this numerator; summing it over $i$ gives the denominator here **and** is what the sectoral price index below aggregates over.

**Sectoral price index.**
$$P_t^{nj} = \Gamma^{nj}\left(\sum_{i=1}^N \left(w_t^i \kappa_t^{nj,ij}\right)^{-\theta^j}\left(A_t^{ij}\right)^{\theta^j}\right)^{-1/\theta^j}$$

In [ ]:
# Discretize the continuum of varieties omega in [0,1] for each region-sector into n_omega grid
# points, and draw each variety's idiosyncratic productivity z^{nj}(omega) i.i.d. from the standard
# Frechet marginal (shape theta^j, scale 1):
#   Phi^{nj}(z) = exp{-(z)^{-theta^j}}
# A^{nj} is the location/scale parameter: actual productivity is A^{nj} * z^{nj}(omega).
#
# Simplified production function: no materials or structures (no xi or gamma cost shares), so
# output of variety omega is its productivity times labor allocated to it:
#   q^{nj}(omega) = z^{nj}(omega) * A^{nj} * l^{nj}(omega)
# Labor is split evenly across varieties within a region-sector (CRS + identical unit cost across
# varieties in the same region-sector means the split doesn't matter for aggregate outcomes):
#   l^{nj}(omega) = L^{nj} / n_omega

omega = range(0, 1, length=n_omega) # variety index grid, per region-sector

z_0 = [rand(Frechet(theta[j], 1)) for n in 1:N, j in 1:J, m in 1:n_omega] # standard Frechet draw per variety

l_0 = [L_0[n,j+1] / n_omega for n in 1:N, j in 1:J, m in 1:n_omega] # labor input per variety, split evenly
                                                                     # (real sector j is market column j+1 in L_0)

q_0 = [z_0[n,j,m] * A_0[n,j] * l_0[n,j,m] for n in 1:N, j in 1:J, m in 1:n_omega]


In [ ]:
# Unit cost of the input bundle: with no materials or structures in this simplified model,
# the cost per efficiency unit of the input bundle is just the wage scaled by the coefficient B:
#   x^{nj} = B^{nj} * w^{nj}

x_0 = B .* w_0


In [ ]:
# Trade shares: the fraction of region n's spending on sector j goods sourced from region i, i.e.
# the probability that region i is the lowest-cost producer for a given variety of sector j:
#   pi^{nj,ij} = (x^{ij} * kappa^{nj,ij})^{-theta^j} * (A^{ij})^{theta^j * gamma^{ij}}
#                / sum_i' (x^{i'j} * kappa^{nj,i'j})^{-theta^j} * (A^{i'j})^{theta^j * gamma^{i'j}}
#
# trade_cost_term[n,j,i] is exactly the numerator: each source region i's contribution to the
# aggregate. Summing it over i gives the denominator here AND is what the sectoral price index
# below aggregates over -- P^{nj} is literally built from the same lowest-cost-producer terms.

trade_cost_term = [
    (x_0[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j] * gamma[i,j])
    for n in 1:N, j in 1:J, i in 1:N
]

pi_0 = [
    trade_cost_term[n,j,i] / sum(trade_cost_term[n,j,:])
    for n in 1:N, j in 1:J, i in 1:N
]


In [ ]:
# Sectoral price index: the same CES-Frechet closed form as before, but now expressed via the
# trade_cost_term sum built above -- P^{nj} aggregates the landed costs of the lowest-cost
# producers across all source regions i, exactly the sum used to normalize the trade shares:
#   P_t^{nj} = Gamma^{nj} * ( sum_i (x^{ij} * kappa^{nj,ij})^{-theta^j} * (A^{ij})^{theta^j*gamma^{ij}} )^{-1/theta^j}

Gamma_const = [SpecialFunctions.gamma((theta[j] + 1 - eta[n,j]) / theta[j])^(1 / (1 - eta[n,j])) for n in 1:N, j in 1:J]

P_0 = [
    Gamma_const[n,j] * sum(trade_cost_term[n,j,:])^(-1/theta[j])
    for n in 1:N, j in 1:J
]

p_0 = P_0 # feeds directly into the household budget constraints below


# Household Problem

This section implements the static half of the household block: given wages and prices, a household's consumption decision has a closed form and doesn't require solving an optimization.

**Ideal price index**, region-specific (the same for every employed household in region $n$, regardless of which sector they work in):
$$P_t^n = \prod_{k=1}^J \left(\frac{P_t^{nk}}{\alpha^k}\right)^{\alpha^k}$$

**Real consumption** of an employed household in $(n,j)$:
$$C_t^{nj} = \frac{w_t^{nj}}{P_t^n}$$
and Cobb-Douglas demand splits nominal spending across the $J$ goods:
$$c_t^{nj,k} = \alpha^k \frac{w_t^{nj}}{P_t^{nk}}$$

**Utility** is logarithmic: $U(C_t^{nj}) = \log(C_t^{nj})$. A non-employed household's consumption is fixed at $b^n$ (no Cobb-Douglas bundle, no price index), so `U_0_mkt` stacks $b^n$ in column 1 (non-employment) alongside $C_0$ in columns 2:M (the $J$ real sectors) — this is the full $N \times M$ flow-utility array the migration/value-function block below needs.

In [ ]:
# Household problem: a worker whose state is region n, employed in sector j, earns wage w_0[n,j]
# and spends it on consumption c[n,j,g] of each good g, with Cobb-Douglas preferences over goods.
# This has a closed form -- no need to solve it as an optimization:
#
# 1) Deflate nominal wage by the ideal price index of the consumption bundle to get aggregate
#    (real) consumption. The ideal price index depends only on the region n (prices faced), not
#    the employment sector j:
#      P_hat^n = prod_g (p_0^{ng} / alpha^g)^{alpha^g}
# 2) Cobb-Douglas shares then split expenditure w_0[n,j] across goods:
#      c^{nj,g} = alpha^g * w_0^{nj} / p_0^{ng}

I_0 = vec(sum(w_0 .* L_0[:, 2:M], dims=2)) # aggregate labor income by region: wage bill over employed
                                            # markets only (L_0[:,2:M] = real-sector employment; the
                                            # non-employment market in column 1 earns no wage)

P_hat = [prod((p_0[n,g] / alpha[g])^alpha[g] for g in 1:J) for n in 1:N] # ideal price index, by region

C_0 = [w_0[n,j] / P_hat[n] for n in 1:N, j in 1:J] # aggregate (real) consumption index

c_0 = [alpha[g] * w_0[n,j] / p_0[n,g] for n in 1:N, j in 1:J, g in 1:J]


In [ ]:
# Utility index for each region-sector state. By the Cobb-Douglas identity,
# prod_g(c^{nj,g})^{alpha^g} == w_0^{nj}/P_hat^n, so this is just the aggregate consumption
# index C_0 already computed above -- kept as a named alias.
U_0 = C_0

# Flow utility over every N x M market (rows = regions, columns = markets, column 1 = non-
# employment). Non-employment consumption is fixed at b^n (no Cobb-Douglas bundle, no price
# index); real sector j's flow utility is U_0[n,j] computed above, living in market column j+1:
U_0_mkt = hcat(b, U_0)


## Migration decision

This subsection implements the dynamic household block's Bellman equation and migration shares (eqs 2–3).

**Bellman equation.** After integrating out i.i.d. Type-I Extreme Value taste shocks over destinations, the expected lifetime value of a household in market $(n,j)$ is:
$$V_t^{nj} = U(C_t^{nj}) + \nu \log\left(\sum_{i=1}^N\sum_{k=0}^J \exp\left(\frac{\beta V_{t+1}^{ik} - \tau^{nj,ik}}{\nu}\right)\right)$$

**Migration shares**, the share of households relocating from $(n,j)$ to $(i,k)$:
$$\mu_t^{nj,ik} = \frac{\exp\left(\dfrac{\beta V_{t+1}^{ik} - \tau^{nj,ik}}{\nu}\right)}{\displaystyle\sum_{m=1}^N\sum_{h=0}^J \exp\left(\dfrac{\beta V_{t+1}^{mh} - \tau^{nj,mh}}{\nu}\right)}$$
Note the log-sum-exp term inside $V_t^{nj}$ is exactly the (unnormalized) denominator of $\mu_t^{nj,\cdot}$ — $V$ and $\mu$ are two views of the same object.

**Why fixed-point iteration.** Wages/prices (hence flow utility) are held fixed here, so this is a *stationary* environment: $V_t = V_{t+1} = V^*$ for every $t$, and $V^*$ is the unique fixed point of the Bellman operator above (a contraction since $\beta<1$, by Blackwell's sufficient conditions). The code finds $V^*$ by value function iteration starting from a flow-utility-only guess, then computes $\mu$ from the converged $V^*$. Once wages depend on the evolving labor distribution (Notebook 3's Sequential Equilibrium section), $V$ and $\mu$ must be recomputed period by period instead.

In [ ]:
# Migration decision: a worker in state (n,j) [region n, market j -- j=0 is non-employment,
# j=1..J is employed in that sector] chooses migration probabilities mu[n,j,i,k] over every
# destination market (i,k) to maximize expected DISCOUNTED FUTURE value net of moving cost,
# subject to a Frechet/type-I EV taste shock over destinations (dispersion nu). The closed form
# is the discrete-choice logit/softmax over destinations:
#   mu^{nj,ik} = exp((beta*V^{ik} - tau^{nj,ik})/nu)
#                / sum_{m,h} exp((beta*V^{mh} - tau^{nj,mh})/nu)
#
# V itself is defined recursively (today's flow utility plus the same logit continuation value):
#   V^{nj} = log(U_0_mkt^{nj}) + nu*log( sum_{i,k} exp((beta*V^{ik} - tau^{nj,ik})/nu) )
# All arrays here are N x M (rows = regions, columns = markets, column 1 = non-employment),
# summing over every (i,k) market pair a household could move to. Wages/prices (hence flow
# utility U_0_mkt) are still held fixed at their Parameters-cell values, so this is a STATIONARY
# environment: rather than a static one-shot placeholder for V, solve the fixed point by value
# function iteration (a contraction since beta<1). Because the environment doesn't change over
# time, the resulting V -- and hence mu -- is the same every period; the forward-looking
# recursion is baked into V itself rather than needing to be re-solved period by period. Once
# wages/prices respond to the evolving labor distribution (production feeding back into w_0/p_0
# each period), V and mu will need to be recomputed period by period against that time-varying
# state.

V = log.(U_0_mkt) # (N,M) initial guess: flow utility only (this was the old static placeholder)
for iter in 1:10_000
    V_next = [
        log(U_0_mkt[n,j]) + nu * log(sum(exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
        for n in 1:N, j in 1:M
    ]
    converged = maximum(abs.(V_next .- V)) < 1e-12
    V = V_next
    converged && break
end

mu_0 = [
    exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) /
    sum(exp((beta*V[m,h] - tau_mig[n,j,m,h]) / nu) for m in 1:N, h in 1:M)
    for n in 1:N, j in 1:M, i in 1:N, k in 1:M
]


## Labor dynamics

This subsection implements the law of motion for the state (eq 4):
$$L_{t+1}^{nj} = \sum_{i=1}^N \sum_{k=0}^J \mu_t^{ik,nj} L_t^{ik}$$
Every household currently in market $(i,k)$ relocates according to the migration shares $\mu_t^{ik,nj}$ computed above; summing their contributions across every origin $(i,k)$ gives next period's mass in $(n,j)$. Because $\mu_0$ was solved as the stationary fixed point above (constant across $t$), it's applied unchanged at every step of this forward simulation — this is only valid because wages/prices never change in this section.

In [ ]:
# Simulate the labor distribution forward using the migration law of motion:
#   L_{t+1}^{nj} = sum_i sum_k mu^{ik,nj} * L_t^{ik}
#
# mu_0[i,k,n,j] is the share moving FROM (i,k) TO (n,j), i.e. mu^{ik,nj}. Every L_path entry is
# N x M (rows = regions, columns = markets, column 1 = non-employment), matching L_0.
#
# mu_0 is now the value-function fixed point (see Migration decision above), not a one-shot flow
# utility placeholder. It's applied unchanged every period here because wages/prices are still
# exogenous constants from the Parameters cell, so the migration environment is stationary and
# mu is correctly constant over time. Once production feeds back into w_0/p_0 each period (so the
# environment is no longer stationary), V and mu will need to be recomputed inside this loop each
# period against that period's state.

L_path = Vector{Matrix{Float64}}()
push!(L_path, L_0)

for t in 1:T
    L_next = [
        sum(mu_0[i,k,n,j] * L_path[t][i,k] for i in 1:N, k in 1:M)
        for n in 1:N, j in 1:M
    ]
    push!(L_path, L_next)
end
